# UrbanFlow AI — Multi-Scale STGCN Kernel Ablation

This notebook compares three controlled configurations: **kernel 3**, **kernel 5**, and **combined kernels 3 + 5**. Only the temporal kernel configuration changes. Dataset splits, seed, model width, optimizer, loss, and epoch limit remain fixed.

Input shape: `(B, 12, 207, 1)`; target/prediction shape: `(B, 3, 207, 1)`. Results and predictions are saved under `/content/experiments/multiscale_ablation/`. The standard proposed-model prediction files are not overwritten.

## Cell 1 — Imports, configuration, and paths

In [5]:
from pathlib import Path
import json, pickle, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED=42; INPUT_STEPS=12; OUTPUT_STEPS=3; NUM_NODES=207
HIDDEN_CHANNELS=32; DROPOUT=0.2; BATCH_SIZE=64
LEARNING_RATE=1e-3; WEIGHT_DECAY=1e-4; EPOCHS=20; GRAD_CLIP=5.0
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
PROJECT_ROOT=Path("/content/urbanAi")
PROCESSED_DIR=PROJECT_ROOT/"data"/"processed"/"METR-LA"
RAW_DIR=PROJECT_ROOT/"data"/"raw"/"METR-LA"
OUTPUT_DIR=Path("/content/experiments/multiscale_ablation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE, "Processed data:", PROCESSED_DIR, "Output:", OUTPUT_DIR)

Device: cuda Processed data: /content/urbanAi/data/processed/METR-LA Output: /content/experiments/multiscale_ablation


## Cell 2 — Load and validate data

In [6]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [7]:
from pathlib import Path

# Check the common project and Google Drive locations.
candidates = [
    Path("/content/urbanAi/data/processed/METR-LA"),
    Path("/content/drive/MyDrive/urbanAi/data/processed/METR-LA"),
    Path("/content/drive/MyDrive/urbanAi"),
    Path("/content")
]

for root in candidates:
    print(f"\nChecking: {root}")
    print("Exists:", root.exists())

    if root.exists():
        for path in root.glob("**/train.npz"):
            print("FOUND TRAIN:", path)

        for path in root.glob("**/val.npz"):
            print("FOUND VALIDATION:", path)

        for path in root.glob("**/test.npz"):
            print("FOUND TEST:", path)


Checking: /content/urbanAi/data/processed/METR-LA
Exists: False

Checking: /content/drive/MyDrive/urbanAi/data/processed/METR-LA
Exists: True
FOUND TRAIN: /content/drive/MyDrive/urbanAi/data/processed/METR-LA/train.npz
FOUND VALIDATION: /content/drive/MyDrive/urbanAi/data/processed/METR-LA/val.npz
FOUND TEST: /content/drive/MyDrive/urbanAi/data/processed/METR-LA/test.npz

Checking: /content/drive/MyDrive/urbanAi
Exists: True
FOUND TRAIN: /content/drive/MyDrive/urbanAi/data/processed/METR-LA/train.npz
FOUND VALIDATION: /content/drive/MyDrive/urbanAi/data/processed/METR-LA/val.npz
FOUND TEST: /content/drive/MyDrive/urbanAi/data/processed/METR-LA/test.npz

Checking: /content
Exists: True


KeyboardInterrupt: 

In [ ]:
from pathlib import Path

PROCESSED_DIR = Path(
    "/content/drive/MyDrive/urbanAi/data/processed/METR-LA"
)

assert (PROCESSED_DIR / "train.npz").exists()
assert (PROCESSED_DIR / "val.npz").exists()
assert (PROCESSED_DIR / "test.npz").exists()

print("Processed dataset path verified:", PROCESSED_DIR)

Processed dataset path verified: /content/drive/MyDrive/urbanAi/data/processed/METR-LA


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/urbanAi")

PROCESSED_DIR = (
    PROJECT_ROOT / "data" / "processed" / "METR-LA"
)

RAW_DIR = (
    PROJECT_ROOT / "data" / "raw" / "METR-LA"
)

OUTPUT_DIR = Path("/content/experiments/multiscale_ablation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert (PROCESSED_DIR / "train.npz").exists()
assert (PROCESSED_DIR / "val.npz").exists()
assert (PROCESSED_DIR / "test.npz").exists()

print("Processed dataset:", PROCESSED_DIR)
print("Raw dataset:", RAW_DIR)
print("Output directory:", OUTPUT_DIR)
print("All processed dataset files found.")

Processed dataset: /content/drive/MyDrive/urbanAi/data/processed/METR-LA
Raw dataset: /content/drive/MyDrive/urbanAi/data/raw/METR-LA
Output directory: /content/experiments/multiscale_ablation
All processed dataset files found.


In [ ]:
adjacency_path = RAW_DIR / "adj_METR-LA" / "adj_METR-LA.pkl"
scaler_path = PROCESSED_DIR / "scaler.npz"

print("Adjacency exists:", adjacency_path.exists())
print("Scaler exists:", scaler_path.exists())

assert adjacency_path.exists(), (
    f"Adjacency matrix not found: {adjacency_path}"
)
assert scaler_path.exists(), (
    f"Scaler not found: {scaler_path}"
)

Adjacency exists: False
Scaler exists: True


AssertionError: Adjacency matrix not found: /content/drive/MyDrive/urbanAi/data/raw/METR-LA/adj_METR-LA/adj_METR-LA.pkl

In [9]:
from pathlib import Path

search_roots = [
    Path("/content/drive/MyDrive/urbanAi"),
    Path("/content/drive/MyDrive")
]

matches = []

for root in search_roots:
    if root.exists():
        for path in root.rglob("*"):
            if path.is_file() and (
                "adj" in path.name.lower()
                and "metr" in path.name.lower()
            ):
                matches.append(path)

print("Adjacency-related files found:", len(matches))

for path in matches:
    print(path)

Adjacency-related files found: 2
/content/drive/MyDrive/urbanAi/data/processed/METR-LA/adj_METR-LA.pkl
/content/drive/MyDrive/urbanAi/data/processed/METR-LA/adj_METR-LA.pkl


In [10]:
from pathlib import Path
import pickle
import numpy as np
import torch

adjacency_path = Path(
    "/content/drive/MyDrive/urbanAi/data/processed/METR-LA/adj_METR-LA.pkl"
)

assert adjacency_path.exists(), f"Missing file: {adjacency_path}"

with open(adjacency_path, "rb") as f:
    adjacency_data = pickle.load(f, encoding="latin1")

if isinstance(adjacency_data, (tuple, list)):
    adjacency_np = np.asarray(adjacency_data[-1], dtype=np.float32)
else:
    adjacency_np = np.asarray(adjacency_data, dtype=np.float32)

assert adjacency_np.shape == (207, 207), (
    f"Expected adjacency shape (207, 207), got {adjacency_np.shape}"
)

degree = adjacency_np.sum(axis=1)
assert np.all(degree > 0), "At least one node has zero degree."

adjacency_np = adjacency_np / degree[:, None]

assert np.allclose(adjacency_np.sum(axis=1), 1.0, atol=1e-5)

A = torch.tensor(
    adjacency_np,
    dtype=torch.float32,
    device=DEVICE
)

print("Adjacency loaded successfully.")
print("Shape:", tuple(A.shape))
print("Device:", A.device)

Adjacency loaded successfully.
Shape: (207, 207)
Device: cuda:0


In [12]:
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# Processed dataset location in Google Drive
PROCESSED_DIR = Path(
    "/content/drive/MyDrive/urbanAi/data/processed/METR-LA"
)

# Verify required files before loading
for filename in ["train.npz", "val.npz", "test.npz"]:
    path = PROCESSED_DIR / filename
    assert path.exists(), f"Dataset file not found: {path}"


class TrafficDataset(Dataset):
    def __init__(self, path):
        with np.load(path) as d:
            self.X = torch.from_numpy(
                d["X"].astype(np.float32)
            )
            self.Y = torch.from_numpy(
                d["Y"].astype(np.float32)
            )

        assert self.X.ndim == 4, self.X.shape
        assert self.Y.ndim == 4, self.Y.shape

        assert tuple(self.X.shape[1:]) == (12, 207, 1), self.X.shape
        assert tuple(self.Y.shape[1:]) == (3, 207, 1), self.Y.shape
        assert len(self.X) == len(self.Y)

        assert torch.isfinite(self.X).all(), "X contains NaN or Inf"
        assert torch.isfinite(self.Y).all(), "Y contains NaN or Inf"

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.Y[i]


train_dataset = TrafficDataset(PROCESSED_DIR / "train.npz")
val_dataset = TrafficDataset(PROCESSED_DIR / "val.npz")
test_dataset = TrafficDataset(PROCESSED_DIR / "test.npz")


def make_loaders(seed):
    generator = torch.Generator().manual_seed(seed)

    return (
        DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            generator=generator
        ),
        DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False
        ),
        DataLoader(
            test_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False
        )
    )


print("Dataset loaded successfully.")
print("Samples:", len(train_dataset), len(val_dataset), len(test_dataset))
print(
    "X sample:", tuple(train_dataset.X.shape[1:]),
    "| Y sample:", tuple(train_dataset.Y.shape[1:])
)

Dataset loaded successfully.
Samples: 23976 3413 6841
X sample: (12, 207, 1) | Y sample: (3, 207, 1)


## Cell 3 — Load adjacency matrix and training-only scaler

In [13]:
from pathlib import Path
import pickle
import numpy as np
import torch

# Adjacency matrix is stored in the processed-data folder.
adj_path = PROCESSED_DIR / "adj_METR-LA.pkl"

assert adj_path.exists(), f"Adjacency file not found: {adj_path}"

with open(adj_path, "rb") as f:
    adj_data = pickle.load(f, encoding="latin1")

adj_np = np.asarray(
    adj_data[-1] if isinstance(adj_data, (tuple, list)) else adj_data,
    dtype=np.float32
)

assert adj_np.shape == (207, 207), adj_np.shape

degree = adj_np.sum(axis=1)
assert np.all(degree > 0), "A graph row has zero degree"

adj_np = adj_np / degree[:, None]

assert np.allclose(
    adj_np.sum(axis=1), 1.0, atol=1e-5
)

A = torch.tensor(
    adj_np,
    dtype=torch.float32,
    device=DEVICE
)

# Load the existing training-only scaler.
scaler_path = PROCESSED_DIR / "scaler.npz"
assert scaler_path.exists(), f"Scaler file not found: {scaler_path}"

with np.load(scaler_path) as s:
    scaler_mean = s["mean"].astype(np.float32).reshape(-1)
    scaler_std = s["std"].astype(np.float32).reshape(-1)

assert scaler_mean.shape == (207,)
assert scaler_std.shape == (207,)
assert np.all(scaler_std > 0)

mean_np = scaler_mean.reshape(1, 1, 207, 1)
std_np = scaler_std.reshape(1, 1, 207, 1)

def inverse_scale(x):
    return x * std_np + mean_np

print("Adjacency and scaler loaded successfully.")
print("Adjacency shape:", tuple(A.shape))
print("Adjacency device:", A.device)
print("Scaler sensors:", len(scaler_mean))

Adjacency and scaler loaded successfully.
Adjacency shape: (207, 207)
Adjacency device: cuda:0
Scaler sensors: 207


## Cell 4 — Define the model

In [14]:
class MultiScaleTemporalConv(nn.Module):
    def __init__(self,in_channels,out_channels,kernels=(3,5),dropout=0.2):
        super().__init__()
        self.branches=nn.ModuleList([nn.Conv2d(in_channels,out_channels,kernel_size=(1,k),padding=(0,k//2)) for k in kernels])
        self.fusion=nn.Conv2d(out_channels*len(kernels),out_channels,kernel_size=1)
        self.activation=nn.ReLU(); self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        x=x.permute(0,3,2,1)  # B,C,N,T
        x=torch.cat([branch(x) for branch in self.branches],dim=1)
        x=self.dropout(self.activation(self.fusion(x)))
        return x.permute(0,3,2,1)  # B,T,N,C

class GraphConvolution(nn.Module):
    def __init__(self,in_channels,out_channels):
        super().__init__(); self.linear=nn.Linear(in_channels,out_channels)
    def forward(self,x,adjacency):
        return self.linear(torch.einsum("nm,btmc->btnc",adjacency,x))

class MultiScaleSTGCNBlock(nn.Module):
    def __init__(self,in_channels,hidden_channels,kernels=(3,5),dropout=0.2):
        super().__init__()
        self.temporal1=MultiScaleTemporalConv(in_channels,hidden_channels,kernels,dropout)
        self.graph=GraphConvolution(hidden_channels,hidden_channels)
        self.temporal2=MultiScaleTemporalConv(hidden_channels,hidden_channels,kernels,dropout)
        self.activation=nn.ReLU()
    def forward(self,x,adjacency):
        x=self.temporal1(x)
        x=self.activation(self.graph(x,adjacency))
        return self.temporal2(x)

class MultiScaleSTGCN(nn.Module):
    def __init__(self,input_steps=12,output_steps=3,input_channels=1,hidden_channels=32,kernels=(3,5),dropout=0.2):
        super().__init__()
        self.block1=MultiScaleSTGCNBlock(input_channels,hidden_channels,kernels,dropout)
        self.block2=MultiScaleSTGCNBlock(hidden_channels,hidden_channels,kernels,dropout)
        self.temporal_head=nn.Linear(input_steps,output_steps)
        self.output_head=nn.Linear(hidden_channels,1)
    def forward(self,x,adjacency):
        x=self.block1(x,adjacency); x=self.block2(x,adjacency)
        x=x.permute(0,2,3,1)   # B,N,C,T
        x=self.temporal_head(x) # B,N,C,H
        x=x.permute(0,3,1,2)  # B,H,N,C
        return self.output_head(x)

## Cell 5 — Metrics

In [15]:
def calculate_metrics(y_true,y_pred,epsilon=1e-5):
    y_true=np.asarray(y_true); y_pred=np.asarray(y_pred)
    mae=np.mean(np.abs(y_true-y_pred)); rmse=np.sqrt(np.mean((y_true-y_pred)**2))
    mask=np.abs(y_true)>epsilon
    mape=(np.mean(np.abs((y_true[mask]-y_pred[mask])/y_true[mask]))*100) if np.any(mask) else float("nan")
    return {"MAE":float(mae),"RMSE":float(rmse),"MAPE":float(mape)}

## Cell 6 — Training/evaluation function for one kernel configuration

In [16]:
def run_experiment(kernels,label,seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    train_loader,val_loader,test_loader=make_loaders(seed)
    model=MultiScaleSTGCN(input_steps=12,output_steps=3,input_channels=1,
        hidden_channels=HIDDEN_CHANNELS,kernels=tuple(kernels),dropout=DROPOUT).to(DEVICE)
    optimizer=torch.optim.Adam(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    criterion=nn.MSELoss(); best_val=float("inf"); best_state=None; best_epoch=None; history=[]
    run_dir=OUTPUT_DIR/label; run_dir.mkdir(parents=True,exist_ok=True)
    for epoch in range(1,EPOCHS+1):
        model.train(); train_losses=[]
        for xb,yb in train_loader:
            xb=xb.to(DEVICE); yb=yb.to(DEVICE); optimizer.zero_grad()
            pred=model(xb,A); loss=criterion(pred,yb); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP); optimizer.step()
            train_losses.append(loss.item())
        model.eval(); val_losses=[]
        with torch.no_grad():
            for xb,yb in val_loader:
                pred=model(xb.to(DEVICE),A); val_losses.append(criterion(pred,yb.to(DEVICE)).item())
        tr=float(np.mean(train_losses)); va=float(np.mean(val_losses))
        history.append({"epoch":epoch,"train_mse":tr,"validation_mse":va})
        print(f"{label} | epoch {epoch:02d}/{EPOCHS} | train={tr:.6f} | val={va:.6f}")
        if va<best_val:
            best_val=va; best_epoch=epoch; best_state=copy.deepcopy(model.state_dict())
    if best_state is None: raise RuntimeError("No best checkpoint selected")
    model.load_state_dict(best_state); model.eval(); pred_batches=[]; target_batches=[]
    with torch.no_grad():
        for xb,yb in test_loader:
            pred_batches.append(model(xb.to(DEVICE),A).cpu().numpy()); target_batches.append(yb.numpy())
    pred_scaled=np.concatenate(pred_batches); true_scaled=np.concatenate(target_batches)
    assert pred_scaled.shape==true_scaled.shape and pred_scaled.shape[1:]==(3,207,1)
    pred_original=inverse_scale(pred_scaled).astype(np.float32)
    true_original=inverse_scale(true_scaled).astype(np.float32)
    np.save(run_dir/"test_predictions.npy",pred_original)
    np.save(run_dir/"test_targets.npy",true_original)
    torch.save({"model_state_dict":model.state_dict(),"kernels":list(kernels),"seed":seed,
        "best_epoch":best_epoch,"best_validation_mse":best_val},run_dir/"best_model.pt")
    with open(run_dir/"training_history.json","w") as f: json.dump(history,f,indent=2)
    result={"configuration":label,"kernels":list(kernels),"seed":seed,"best_epoch":int(best_epoch),
        "best_validation_mse":float(best_val),**calculate_metrics(true_original,pred_original)}
    for h in range(3):
        for metric,value in calculate_metrics(true_original[:,h],pred_original[:,h]).items():
            result[f"{(h+1)*5}min_{metric}"]=value
    with open(run_dir/"metrics.json","w") as f: json.dump(result,f,indent=2)
    print("Finished",label,"overall metrics:",{k:result[k] for k in ("MAE","RMSE","MAPE")})
    return result

In [17]:
import sys
print("Python:", sys.executable)

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: /usr/bin/python3
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [18]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [19]:
print("HELLO — NOTEBOOK IS RUNNING")
2 + 2

HELLO — NOTEBOOK IS RUNNING


4

## Cell 7 — Run kernel 3, kernel 5, and combined 3 + 5

In [21]:
configurations=[("kernel_3",(3,)),("kernel_5",(5,)),("kernels_3_5",(3,5))]
results=[]
for label,kernels in configurations:
    print("\n"+"="*70+f"\nRunning {label}: kernels={kernels}\n"+"="*70)
    results.append(run_experiment(kernels,label,seed=SEED))
results_df=pd.DataFrame(results)
results_df.to_csv(OUTPUT_DIR/"ablation_results.csv",index=False)
with open(OUTPUT_DIR/"ablation_results.json","w") as f: json.dump(results,f,indent=2)
print("Saved comparison to",OUTPUT_DIR/"ablation_results.csv")


Running kernel_3: kernels=(3,)
kernel_3 | epoch 01/20 | train=0.452250 | val=0.354074
kernel_3 | epoch 02/20 | train=0.347127 | val=0.332379
kernel_3 | epoch 03/20 | train=0.335031 | val=0.322815
kernel_3 | epoch 04/20 | train=0.325997 | val=0.323805
kernel_3 | epoch 05/20 | train=0.317139 | val=0.333207
kernel_3 | epoch 06/20 | train=0.313517 | val=0.348215
kernel_3 | epoch 07/20 | train=0.310237 | val=0.326255
kernel_3 | epoch 08/20 | train=0.305612 | val=0.305833
kernel_3 | epoch 09/20 | train=0.303794 | val=0.303681
kernel_3 | epoch 10/20 | train=0.301312 | val=0.307478
kernel_3 | epoch 11/20 | train=0.298068 | val=0.304640
kernel_3 | epoch 12/20 | train=0.299604 | val=0.303181
kernel_3 | epoch 13/20 | train=0.295880 | val=0.303115
kernel_3 | epoch 14/20 | train=0.293976 | val=0.303168
kernel_3 | epoch 15/20 | train=0.290999 | val=0.316098
kernel_3 | epoch 16/20 | train=0.292142 | val=0.301126
kernel_3 | epoch 17/20 | train=0.289164 | val=0.302960
kernel_3 | epoch 18/20 | train=0.

## Cell 8 — Compare overall test metrics

In [22]:
display(results_df[["configuration","kernels","best_epoch","best_validation_mse","MAE","RMSE","MAPE"]].sort_values("MAE").reset_index(drop=True))

,configuration,kernels,best_epoch,best_validation_mse,MAE,RMSE,MAPE
0,kernel_5,[5],20,0.298733,6.216079,11.251278,13.432282
1,kernel_3,[3],18,0.298671,6.255377,11.097375,13.056900
2,kernels_3_5,"[3, 5]",13,0.301801,6.437300,11.311906,14.092523


## Cell 9 — Compare forecast horizons

In [23]:
cols=["configuration"]
for horizon in (5,10,15): cols += [f"{horizon}min_MAE",f"{horizon}min_RMSE",f"{horizon}min_MAPE"]
display(results_df[cols])

,configuration,5min_MAE,5min_RMSE,5min_MAPE,10min_MAE,10min_RMSE,10min_MAPE,15min_MAE,15min_RMSE,15min_MAPE
0,kernel_3,5.826569,10.288802,12.273225,6.262167,11.126056,13.052246,6.677396,11.823977,13.845253
1,kernel_5,5.859101,10.473112,12.757470,6.210337,11.277000,13.414435,6.578798,11.954792,14.124931
2,kernels_3_5,6.067410,10.538600,13.372686,6.418780,11.337696,14.069679,6.825719,12.011343,14.835198


## Cell 10 — Verify saved outputs

In [24]:
for label,_ in configurations:
    folder=OUTPUT_DIR/label
    pred=np.load(folder/"test_predictions.npy"); true=np.load(folder/"test_targets.npy")
    assert pred.shape==true.shape and pred.shape[1:]==(3,207,1)
    assert np.isfinite(pred).all() and np.isfinite(true).all()
    print(f"{label}: predictions={pred.shape}; targets={true.shape}; verified")
print("All outputs are under",OUTPUT_DIR)

kernel_3: predictions=(6841, 3, 207, 1); targets=(6841, 3, 207, 1); verified
kernel_5: predictions=(6841, 3, 207, 1); targets=(6841, 3, 207, 1); verified
kernels_3_5: predictions=(6841, 3, 207, 1); targets=(6841, 3, 207, 1); verified
All outputs are under /content/experiments/multiscale_ablation


## Interpretation

- Lower MAE, RMSE, and MAPE indicate lower error under the corresponding metric.
- Do not assume the combined model must perform best; report the observed results honestly.
- This is a single-seed, single-split comparison. For stronger research claims, repeat with multiple seeds and report variability.
- The temporal kernel values 3 and 5 refer to time steps, not sensor/node counts.